# BFCL-India Fine-Tuning v2

**Changes from v1:**
- Eval monitoring: eval_strategy=steps, eval_steps=50, load_best_model_at_end=True
- Data mix: 50% xLAM (general shape) + 30% Indian + 20% rest (was 99.6% Indian)
- LR: 1e-4 (was 2e-4) — slower learning, less overfitting
- Warmup: 20 steps (was 100) — less wasted on 1K-step training
- 3 epochs (was 2) — more passes with proper early stopping via eval loss

**Expected:** Val loss tracks train loss, gap < 0.15 at end (v1 had gap ~0.4+)

In [ ]:
!pip install -qq unsloth
!nvidia-smi

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

from unsloth import FastLanguageModel
import torch

MAX_SEQ = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    token=os.environ["HF_TOKEN"],
)
print("Model loaded OK")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("LoRA applied OK")

In [ ]:
from datasets import load_dataset

# v2 dataset: 50% xLAM + 30% Indian + 20% rest
ds = load_dataset("bhavjeetsingh2912/toolcaller-train-mix-v2")
print(f"Train: {len(ds['train'])}, Val: {len(ds['validation'])}")

def format_for_qwen(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_ds = ds["train"].map(format_for_qwen, remove_columns=ds["train"].column_names)
val_ds = ds["validation"].map(format_for_qwen, remove_columns=ds["validation"].column_names)
print("Data formatted OK")
print(train_ds[0]["text"][:300])

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        output_dir="outputs",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=20,
        num_train_epochs=3,
        learning_rate=1e-4,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        logging_steps=10,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        max_seq_length=MAX_SEQ,
        dataset_text_field="text",
        packing=False,
        ddp_find_unused_parameters=False,
    ),
)

trainer.train()
print("Training complete!")

In [ ]:
# Plot train vs eval loss — the key diagnostic
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_loss = [(x["step"], x["loss"]) for x in log_history if "loss" in x]
eval_loss = [(x["step"], x["eval_loss"]) for x in log_history if "eval_loss" in x]

if train_loss and eval_loss:
    plt.figure(figsize=(10, 6))
    plt.plot([s for s, _ in train_loss], [l for _, l in train_loss], label="train_loss")
    plt.plot([s for s, _ in eval_loss], [l for _, l in eval_loss], label="eval_loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("v2 Training: Train vs Eval Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig("loss_curve_v2.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved loss_curve_v2.png")
    
    # Check overfitting
    final_train = train_loss[-1][1]
    final_eval = eval_loss[-1][1]
    gap = final_eval - final_train
    print(f"\nFinal train_loss: {final_train:.4f}")
    print(f"Final eval_loss:  {final_eval:.4f}")
    print(f"Gap:              {gap:.4f}")
    if gap > 0.2:
        print("WARNING: Gap > 0.2 — overfitting detected. Consider earlier checkpoint.")
    else:
        print("OK: Gap < 0.2 — acceptable generalization.")
else:
    print("No eval loss recorded — check eval_strategy setting.")

In [ ]:
model.save_pretrained_merged(
    "toolcaller-qwen-3b-v2",
    tokenizer,
    save_method="merged_16bit",
)
print("Model saved locally")

In [ ]:
model.push_to_hub_merged(
    "bhavjeetsingh2912/toolcaller-qwen-3b-v2",
    tokenizer,
    save_method="merged_16bit",
    token=os.environ["HF_TOKEN"],
)
print("Pushed to HuggingFace!")

In [ ]:
!nvidia-smi